## Desafio D — Sistema de Registro de Preços

### Problema

Quais características diferenciam contratações realizadas com e sem Sistema de Registro de Preços?

### Possíveis perguntas

- O SRP é mais frequente em determinados tipos de contratação?
- Existem diferenças nos valores das contratações?
- Determinados órgãos utilizam SRP proporcionalmente mais do que outros?

### Variável de interesse

Quando disponível:

```text
srp
```

### Possíveis análises

- proporções;
- tabelas cruzadas;
- comparação de valores;
- teste qui-quadrado.


documentação API: https://dadosabertos.compras.gov.br/swagger-ui/index.html


In [105]:
#!pip install requests pandas matplotlib -q

In [106]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [107]:
BASE_URL = "https://dadosabertos.compras.gov.br"

ENDPOINT_CONTRATACOES = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"

url = BASE_URL + ENDPOINT_CONTRATACOES

print(url)

https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133


In [108]:
modalidades = [5, 6, 8, 9] # Pegar todas as modalidades de licitação, mas para fins de teste, vamos pegar apenas as principais.
todos_registros = []

In [109]:
def extrair_registros(json_resposta):
    if isinstance(json_resposta, list):
        return json_resposta

    if not isinstance(json_resposta, dict):
        return []

    for chave in ["resultado", "resultados", "data", "content"]:
        if chave in json_resposta and isinstance(json_resposta[chave], list):
            return json_resposta[chave]

    return []



In [110]:

for modalidade in modalidades:
    params = {
        "pagina": 1,
        "tamanhoPagina": 50,
        "dataPublicacaoPncpInicial": "2025-01-01",
        "dataPublicacaoPncpFinal": "2025-12-31",
        "codigoModalidade": modalidade
    }

    resposta = requests.get(
        url,
        params=params,
        timeout=60
        )

    if resposta.status_code == 200:
        dados = resposta.json()
        registros = extrair_registros(dados)
        todos_registros.extend(registros)
        print(
            f"Modalidade {modalidade}: {len(registros)} registros coletados."
        )



Modalidade 5: 50 registros coletados.
Modalidade 6: 50 registros coletados.
Modalidade 8: 0 registros coletados.
Modalidade 9: 0 registros coletados.


In [111]:
resposta = requests.get(
    url,
    params=params,
    timeout=60
)

print("Status HTTP:", resposta.status_code)
print("URL consultada:", resposta.url)

Status HTTP: 200
URL consultada: https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?pagina=1&tamanhoPagina=50&dataPublicacaoPncpInicial=2025-01-01&dataPublicacaoPncpFinal=2025-12-31&codigoModalidade=9


In [112]:
def extrair_registros(json_resposta):
    if isinstance(json_resposta, list):
        return json_resposta

    if not isinstance(json_resposta, dict):
        return []

    for chave in ["resultado", "resultados", "data", "content"]:
        if chave in json_resposta and isinstance(json_resposta[chave], list):
            return json_resposta[chave]

    return []


registros = extrair_registros(dados)

print("Quantidade de registros encontrados:", len(registros))

Quantidade de registros encontrados: 0


In [113]:
print("JSON retornado:", registros)
df = pd.json_normalize(registros)
df.head()

JSON retornado: []


""


In [117]:
#for i in todos_registros:
    #print(i)

In [115]:
import pandas as pd

# 1. Converte a sua variável 'registros' em um DataFrame do Pandas
df = pd.DataFrame(todos_registros)

# Lista com as colunas essenciais para o seu tema
colunas_chave = [
    'srp',
    'numeroCompra',
    'orgaoEntidadeRazaoSocial',
    'modalidadeNome',
    'objetoCompra',
    'valorTotalEstimado',
    'valorTotalHomologado'
]

# Filtra apenas o subconjunto de colunas
df_resumido = df[colunas_chave]

# Separa com e sem SRP
df_com_srp = df_resumido[df_resumido['srp'] == 1]
df_sem_srp = df_resumido[df_resumido['srp'] == 0]

print("=== CONTRATAÇÕES COM SRP ===")
display(df_com_srp.head())

print("\n=== CONTRATAÇÕES SEM SRP ===")
display(df_sem_srp.head())

df_resumido.head(10)

=== CONTRATAÇÕES COM SRP ===


,srp,numeroCompra,orgaoEntidadeRazaoSocial,modalidadeNome,objetoCompra,valorTotalEstimado,valorTotalHomologado
0,True,90039,COMANDO DO EXERCITO,Pregão - Eletrônico,Eventual contratação de serviço de lavadeira c...,49000.00,40950.00
1,True,90001,FUNDACAO UNIVERSIDADE DO AMAZONAS,Pregão - Eletrônico,Registro de preço para eventual contratação de...,17882599.86,NaN
2,True,90088,SECRETARIA DA FAZENDA,Pregão - Eletrônico,Aquisição de material de consumo (água mineral...,1509105.93,940633.95
3,True,90025,COMANDO DA MARINHA,Pregão - Eletrônico,Registro de Preços para eventual aquisição de ...,100038.00,93877.00
4,True,90001,ESTADO DE TOCANTINS,Pregão - Eletrônico,Registro de Preços para eventual aquisição de ...,29619.90,18814.30



=== CONTRATAÇÕES SEM SRP ===


,srp,numeroCompra,orgaoEntidadeRazaoSocial,modalidadeNome,objetoCompra,valorTotalEstimado,valorTotalHomologado
5,False,90025,SAO PAULO SECRETARIA DA ADMINISTRACAO PENITENC...,Pregão - Eletrônico,Contratação de serviço especializado para Manu...,238659.90,178994.93
9,False,91402,ESTADO DO CEARA,Pregão - Eletrônico,O objeto da licitação é a aquisição de MATERIA...,1100000.00,898200.00
10,False,90001,UNIVERSIDADE ESTADUAL DE CIENCIAS DA SAUDE DE ...,Pregão - Eletrônico,Aquisição de material médico hospitalar e afins.,1032491.54,NaN
12,False,90003,MUNICIPIO DE RIO DE JANEIRO,Pregão - Eletrônico,Prestação de Serviços de locação de torres de ...,0.00,NaN
13,False,90098,MUNICIPIO DE LEOPOLDINA,Pregão - Eletrônico,"Contratação de empresa através de Pregão, na f...",186350.00,NaN


,srp,numeroCompra,orgaoEntidadeRazaoSocial,modalidadeNome,objetoCompra,valorTotalEstimado,valorTotalHomologado
0,True,90039,COMANDO DO EXERCITO,Pregão - Eletrônico,Eventual contratação de serviço de lavadeira c...,49000.00,40950.00
1,True,90001,FUNDACAO UNIVERSIDADE DO AMAZONAS,Pregão - Eletrônico,Registro de preço para eventual contratação de...,17882599.86,NaN
2,True,90088,SECRETARIA DA FAZENDA,Pregão - Eletrônico,Aquisição de material de consumo (água mineral...,1509105.93,940633.95
3,True,90025,COMANDO DA MARINHA,Pregão - Eletrônico,Registro de Preços para eventual aquisição de ...,100038.00,93877.00
4,True,90001,ESTADO DE TOCANTINS,Pregão - Eletrônico,Registro de Preços para eventual aquisição de ...,29619.90,18814.30
5,False,90025,SAO PAULO SECRETARIA DA ADMINISTRACAO PENITENC...,Pregão - Eletrônico,Contratação de serviço especializado para Manu...,238659.90,178994.93
6,True,2,MUNICIPIO DE RORAINOPOLIS,Pregão - Presencial,Registro de preços para futura e eventual pres...,8889800.00,NaN
7,True,90111,UNIVERSIDADE FEDERAL DE SANTA MARIA,Pregão - Eletrônico,Registro de Preços para Aquisição de Equipamen...,335775.16,158528.55
8,True,91403,ESTADO DO CEARA,Pregão - Eletrônico,O objeto da licitação é o Registro de Preço pa...,3801877.99,2772720.00
9,False,91402,ESTADO DO CEARA,Pregão - Eletrônico,O objeto da licitação é a aquisição de MATERIA...,1100000.00,898200.00


In [116]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao = df_resumido.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

# Renomeia os índices para facilitar a leitura no relatório
df_qnt_modalidade_orgao.index = ['Sem SRP', 'Com SRP']
display(df_qnt_modalidade_orgao)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,2,46,73
Com SRP,2,18,27
